In [ ]:
# ============================================================
# File: 02_component2_policy_alignment_mapping.py
#
# Purpose:
#   Component 2 — Interpret AI_Risk_Intel in governance context,
#   align to NIST controls and enterprise internal rule sets,
#   compute risk signal, enterprise posture, and create structured
#   Governance_Action_Packets for policy decisions.
#
# Inputs:
#   outputs/intel_objects/ai_risk_intel_lightgbm_m2.jsonl
#   config/NIST_SP-800-53_rev5_catalog_load.csv
#   config/enterprise_controls.json
#   config/asset_criticality.json
#
# Outputs:
#   outputs/intel_objects/governance_action_packets_v3.jsonl
#   outputs/intel_objects/governance_action_packets_v3.csv
# ============================================================

import os, json, csv, logging
from pathlib import Path
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "intel_objects"
CONFIG_DIR   = PROJECT_ROOT / "config"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(OUTPUT_DIR / "component2_policy_mapping.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("component2")

# ------------------------------------------------------------
# 1. Load supporting data (frameworks, enterprise controls)
# ------------------------------------------------------------
def load_controls():
    nist = pd.read_csv(CONFIG_DIR / "NIST_SP-800-53_rev5_catalog_load.csv").fillna("")
    enterprise = json.loads((CONFIG_DIR / "enterprise_controls.json").read_text())
    assets = json.loads((CONFIG_DIR / "asset_criticality.json").read_text())
    return nist, enterprise, assets

# ------------------------------------------------------------
# 2. Compute governance packet from AI intel + internal context
# ------------------------------------------------------------
def build_governance_packets():
    ai_file = OUTPUT_DIR / "ai_risk_intel_lightgbm_m2.jsonl"
    nist, enterprise, assets = load_controls()
    lines = [json.loads(l) for l in ai_file.read_text().splitlines() if l.strip()]
    packets = []

    for rec in lines:
        sev = rec["predicted_severity"]
        conf = max(rec["confidence_distribution"].values())
        asset_weight = assets.get("default", 0.5)
        top_feats = rec["key_features"]
        matched_ctrls = []
        for _, row in nist.iterrows():
            if any(t in row["control_text"].lower() for t in [f.lower() for f in top_feats]):
                cid = row["identifier"]
                ent = enterprise.get(cid, {})
                eff = ent.get("effectiveness_rating", 0.7)
                impl = ent.get("implementation_status", "unknown")
                risk_signal = round(conf * (1 - eff) * asset_weight, 3)
                posture = "Compliant" if risk_signal < 0.25 else "At Risk" if risk_signal < 0.6 else "Non-Compliant"
                matched_ctrls.append({
                    "control_id": cid,
                    "control_name": row["name"],
                    "enterprise_impl_status": impl,
                    "enterprise_effectiveness": eff,
                    "risk_signal": risk_signal,
                    "posture": posture
                })
        requires_human = (sev == "High") or (conf < 0.85) or any(c["posture"] == "Non-Compliant" for c in matched_ctrls)
        packet = {
            "intel_id": rec["intel_id"],
            "predicted_severity": sev,
            "confidence": conf,
            "recommended_actions": "Prioritize remediation in next sprint; verify through retest."
                if sev == "Medium" else "Emergency patch; escalate to security owner.",
            "controls": matched_ctrls,
            "referenced_framework": "NIST SP 800-53 Rev. 5",
            "explanation": f"Severity '{sev}' mapped to {len(matched_ctrls)} controls via features {top_feats}.",
            "requires_human": requires_human,
            "timestamp": datetime.utcnow().isoformat()
        }
        packets.append(packet)
    return packets

# ------------------------------------------------------------
# 3. Save results
# ------------------------------------------------------------
def save_packets(packets):
    out_jsonl = OUTPUT_DIR / "governance_action_packets_v3.jsonl"
    out_csv   = OUTPUT_DIR / "governance_action_packets_v3.csv"
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for p in packets:
            f.write(json.dumps(p) + "\n")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        dw = csv.DictWriter(f, fieldnames=["intel_id","predicted_severity","confidence",
                                           "requires_human","timestamp","referenced_framework"])
        dw.writeheader()
        dw.writerows(packets)
    print(f"Saved {len(packets)} governance packets → {out_jsonl}")

if __name__ == "__main__":
    logger.info("Starting Component 2 pipeline.")
    packets = build_governance_packets()
    save_packets(packets)
    logger.info("Component 2 completed successfully.")

In [2]:
# ============================================================
# File: 02_component2_policy_alignment_v3.py
#
# Description:
#   Component 2 — Policy Alignment and Explainable Decision Mapping (v3)
#   Interprets AI_Risk_Intel as governance decision candidates, maps to NIST,
#   consults enterprise internal ruleset, applies asset criticality, computes
#   composite score and proactive risk_signal, then emits Governance_Action_Packets.
#
# Inputs:
#   - PROJECT_ROOT/outputs/intel_objects/ai_risk_intel_*.jsonl
#   - PROJECT_ROOT/NIST_SP-800-53_rev5_catalog_load.csv
#   - PROJECT_ROOT/config/enterprise_controls.json
#   - PROJECT_ROOT/config/asset_criticality.json
#
# Outputs:
#   - outputs/intel_objects/governance_action_packets_v3.jsonl
#   - outputs/intel_objects/governance_action_packets_v3.csv
#   - outputs/reports/governance_enterprise_summary_v3.html
#   - outputs/logs/component2_policy_alignment_v3.log
#
# Notes:
#   - Pure local, reproducible.  No network calls.
#   - Mapping uses lexical overlap + curated CWE/CPE hints.
#     A transformer-based mapping can be plugged in later.
# ============================================================

from __future__ import annotations

import os, re, csv, json, logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Dynamic project root detection (portable)
# ------------------------------------------------------------
def find_project_root() -> Path:
    """Locate the project root dynamically for macOS M2, Linux, or Colab."""
    candidates = [
        Path.cwd(),
        Path.home() / "Research" / "AI_GRC_Project",
        Path("/content/AI_GRC_Project"),  # Colab
    ]
    for c in candidates:
        if (c / "outputs").exists() or (c / "data").exists() or (c / "config").exists():
            return c.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
PATH_NIST    = PROJECT_ROOT / "NIST_SP-800-53_rev5_catalog_load.csv"
DIR_OUTPUT   = PROJECT_ROOT / "outputs" / "intel_objects"
DIR_REPORTS  = PROJECT_ROOT / "outputs" / "reports"
DIR_LOGS     = PROJECT_ROOT / "outputs" / "logs"
DIR_CONFIG   = PROJECT_ROOT / "config"

for d in (DIR_OUTPUT, DIR_REPORTS, DIR_LOGS, DIR_CONFIG):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(DIR_LOGS / "component2_policy_alignment_v3.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("component2_v3")

# ------------------------------------------------------------
# 2. Constants / Policy Weights
# ------------------------------------------------------------
SEVERITY_ACTIONS = {
    "Low":    "Patch in next maintenance window; verify via routine scan (≤ 30 days).",
    "Medium": "Prioritize remediation in the next sprint; retest on completion (≤ 14 days).",
    "High":   "Emergency patch/mitigation; isolate if needed; retest (≤ 72 hours).",
}
SEVERITY_WEIGHTS = {"Low": 0.8, "Medium": 1.0, "High": 1.2}

# Mappings and hints
CWE_TO_NIST = {
    "CWE-79":  ["SI-10", "SC-7", "RA-5"],
    "CWE-89":  ["SI-10", "RA-5", "SA-11"],
    "CWE-352": ["AC-6", "IA-2", "SI-10"],
    "CWE-22":  ["SC-7", "SI-10"],
    "CWE-78":  ["SI-10", "SI-3", "RA-5"],
    "CWE-119": ["SI-2", "SA-11", "RA-5"],
    "CWE-287": ["IA-2", "AC-6"],
    "CWE-285": ["AC-3", "AC-6"],
    "CWE-798": ["IA-5", "SC-12", "CM-6"],
    "CWE-200": ["SC-7", "SC-28", "CM-2"],
}
CWE_HINTS = {
    "xss": ["SI-10", "SC-7", "RA-5"],
    "sql injection": ["SI-10", "RA-5", "SA-11"],
    "command injection": ["SI-10", "SI-3", "RA-5"],
    "buffer overflow": ["SI-2", "SA-11"],
    "csrf": ["AC-6", "IA-2", "SI-10"],
    "authentication": ["IA-2", "AC-6"],
    "authorization": ["AC-3", "AC-6"],
    "path traversal": ["SC-7", "SI-10"],
}
CPE_HINTS = {
    "microsoft": ["AC", "SC", "SI"],
    "wordpress": ["SI", "AC"],
    "apache": ["SC", "SI"],
    "linux": ["SI", "CM"],
    "cisco": ["SC", "AC"],
}
NIST_TO_ISO = {
    "AC": ["ISO 27001 A.9 (Access Control)"],
    "IA": ["ISO 27001 A.9.4 (Access Control)"],
    "SC": ["ISO 27001 A.13 (Network Security)"],
    "SI": ["ISO 27001 A.12 (Operations Security)"],
    "RA": ["ISO 27001 A.12.6.1 (Vulnerability Management)"],
    "CM": ["ISO 27001 A.12 (Change Management)"],
    "SA": ["ISO 27001 A.14 (Secure Development)"],
}
NIST_TO_PCI = {
    "AC": ["PCI DSS Req. 7 (Access Control)"],
    "IA": ["PCI DSS Req. 8 (Identify/Authenticate)"],
    "SC": ["PCI DSS Req. 1 (Network Security)"],
    "SI": ["PCI DSS Req. 5/6 (Malware/Hardening)"],
    "RA": ["PCI DSS Req. 11 (Testing/Scanning)"],
    "CM": ["PCI DSS Req. 6 (Change Management)"],
    "SA": ["PCI DSS Req. 6 (Secure SDLC)"],
}

# ------------------------------------------------------------
# 3. Loaders
# ------------------------------------------------------------
def load_latest_ai_risk_intel() -> List[Dict[str, Any]]:
    cands = sorted(DIR_OUTPUT.glob("ai_risk_intel_*.jsonl"))
    if not cands:
        raise FileNotFoundError(f"No ai_risk_intel_*.jsonl found in {DIR_OUTPUT}")
    latest = max(cands, key=lambda p: p.stat().st_mtime)
    rows = [json.loads(l) for l in latest.read_text(encoding="utf-8").splitlines() if l.strip()]
    logger.info("Loaded %d AI_Risk_Intel records from %s", len(rows), latest.name)
    return rows

def load_nist_catalog(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"NIST CSV not found: {path}")
    df = pd.read_csv(path)
    req = ["identifier","name","control_text","discussion","related"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"NIST CSV missing columns: {miss}")
    df = df[req].copy().fillna("")
    def norm(s): return re.sub(r"[^a-z0-9]+"," ",s.lower()).strip()
    df["blob"] = (df["identifier"].map(norm)+" "+df["name"].map(norm)+" "+
                  df["control_text"].map(norm)+" "+df["discussion"].map(norm)+" "+
                  df["related"].map(norm))
    df["family"] = df["identifier"].str.extract(r"^([A-Z]{2})", expand=False).fillna("")
    logger.info("Loaded %d NIST controls", len(df))
    return df

def load_enterprise_controls(path: Path) -> Dict[str, Dict[str, Any]]:
    if not path.exists():
        logger.warning("enterprise_controls.json not found; continuing without enterprise context.")
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    ctrls = data.get("controls", [])
    return {c["control_id"]: c for c in ctrls if "control_id" in c}

def load_asset_weight(path: Path) -> float:
    if not path.exists():
        logger.warning("asset_criticality.json not found; default weight = 0.7")
        return 0.7
    data = json.loads(path.read_text(encoding="utf-8"))
    assets = data.get("assets", [])
    if not assets:
        return 0.7
    return float(np.mean([float(a.get("criticality", 0.7)) for a in assets]))

# ------------------------------------------------------------
# 4. Helpers / Scoring
# ------------------------------------------------------------
def tok(s: str) -> List[str]:
    return re.sub(r"[^a-z0-9]+"," ",str(s).lower()).split()

def extract_cwes(rec: Dict[str, Any]) -> List[str]:
    vals=[]
    for k in ("cwe","cwe_id","cwe_ids","cwes","cwe_list"):
        v=rec.get(k)
        if not v: continue
        if isinstance(v,str): vals+=re.findall(r"CWE-\d+",v.upper())
        elif isinstance(v,list):
            for it in v: vals+=re.findall(r"CWE-\d+",str(it).upper())
    seen=set(); out=[]
    for cid in vals:
        if cid not in seen: seen.add(cid); out.append(cid)
    return out

def extract_cpe_tokens(rec: Dict[str, Any]) -> List[str]:
    allv=[]
    for k in ("cpe","cpe23","cpe_list","cpe_matches","cpe_uris"):
        v=rec.get(k)
        if not v: continue
        allv += [v] if isinstance(v,str) else [str(x) for x in v]
    toks=set()
    for c in allv: toks.update(tok(c))
    return list(toks)

def families_from_cwe_features(cwes: List[str], features: List[str]) -> List[str]:
    fams=[]
    for cid in cwes: fams+=CWE_TO_NIST.get(cid,[])
    if not fams and features:
        joined=" ".join(features).lower()
        for hint,ctrls in CWE_HINTS.items():
            if hint in joined: fams+=ctrls
    out,seen=[],set()
    for ctrl in fams:
        m=re.match(r"^([A-Z]{2})",ctrl)
        if m and m.group(1) not in seen:
            seen.add(m.group(1)); out.append(m.group(1))
    return out[:4]

def families_from_cpe_tokens(tokens: List[str]) -> List[str]:
    fams,seen=[],set()
    for t in tokens:
        if t in CPE_HINTS:
            for f in CPE_HINTS[t]:
                if f not in seen:
                    seen.add(f); fams.append(f)
    return fams[:4]

class NISTIndex:
    def __init__(self, df: pd.DataFrame): self.df=df
    def search_by_tokens(self, tokens: List[str], top_k: int=30) -> List[Dict[str,Any]]:
        q={t for t in tokens if t}
        if not q: return []
        scored=[]
        for i,row in self.df.iterrows():
            words=set(row["blob"].split())
            ov=len(q & words)/max(1,len(q))
            if ov>0: scored.append((ov,i))
        scored.sort(reverse=True)
        idx=[i for _,i in scored[:top_k]]
        return self.df.iloc[idx].to_dict(orient="records")
    def by_family(self, fams: List[str], top_k:int=40)->List[Dict[str,Any]]:
        if not fams: return []
        sub=self.df[self.df["family"].isin([f.upper() for f in fams])]
        return sub.head(top_k).to_dict(orient="records")

def composite_score(kw_overlap,fam_bonus,cwe_match,cpe_match,confidence,asset_weight,sev_weight):
    base=(0.4*kw_overlap)+(0.2*fam_bonus)+(0.2*cwe_match)+(0.2*cpe_match)
    conf_scale=0.5+0.5*max(0,min(1,confidence))
    asset_scale=0.6+0.4*max(0,min(1,asset_weight))
    return round(base*conf_scale*asset_scale*sev_weight,3)

def risk_signal(confidence,effectiveness,asset_weight):
    return round(confidence*(1.0-effectiveness)*asset_weight,3)

def classify_posture_by_risk_signal(sig):
    if sig<0.25: return "Compliant"
    if sig<0.60: return "At Risk"
    return "Non-Compliant"

def requires_human(severity,confidence,asset_weight,top_score,any_noncompliant):
    if severity.lower()=="high": return True
    if confidence<0.85: return True
    if any_noncompliant: return True
    if asset_weight>=0.85 and top_score>=0.60: return True
    return False

def enrich_iso_pci(nist_ids: List[str]) -> Dict[str, List[str]]:
    fams=set()
    for cid in nist_ids:
        m=re.match(r"^([A-Z]{2})",cid)
        if m: fams.add(m.group(1))
    iso=[r for f in fams for r in NIST_TO_ISO.get(f,[])]
    pci=[r for f in fams for r in NIST_TO_PCI.get(f,[])]
    return {"ISO27001":list(dict.fromkeys(iso)),"PCI-DSS":list(dict.fromkeys(pci))}

# ------------------------------------------------------------
# 5. Main Pipeline
# ------------------------------------------------------------
def main()->None:
    ai_intel=load_latest_ai_risk_intel()
    nist_df=load_nist_catalog(PATH_NIST)
    idx=NISTIndex(nist_df)
    ent_ctrl=load_enterprise_controls(DIR_CONFIG/"enterprise_controls.json")
    asset_w=load_asset_weight(DIR_CONFIG/"asset_criticality.json")

    packets=[]
    for rec in ai_intel:
        intel_id=rec.get("intel_id","")
        severity=(rec.get("predicted_severity") or "Medium").title()
        sev_w=SEVERITY_WEIGHTS.get(severity,1.0)
        dist=rec.get("prediction_distribution") or {}
        confidence=float(max(dist.values())) if dist else 1.0
        features=rec.get("key_influential_features") or rec.get("key_features") or []
        desc=rec.get("description") or rec.get("summary") or ""

        cwes=extract_cwes(rec)
        cpe_toks=extract_cpe_tokens(rec)
        fam_cwe=families_from_cwe_features(cwes,features)
        fam_cpe=families_from_cpe_tokens(cpe_toks)
        fams=list(dict.fromkeys([*fam_cwe,*fam_cpe]))

        q_tokens=set()
        for f in features: q_tokens.update(tok(f))
        for cw in cwes: q_tokens.update(tok(cw))
        for t in cpe_toks: q_tokens.add(t)
        q_tokens.update(tok(desc)[:16])
        q_tokens=list(q_tokens)

        cand=idx.search_by_tokens(q_tokens,top_k=40)
        if fams: cand+=idx.by_family(fams,top_k=40)

        seen=set(); uniq=[]
        for c in cand:
            cid=c["identifier"]
            if cid not in seen:
                seen.add(cid); uniq.append(c)

        scored=[]
        qset,famset=set(q_tokens),set(fams)
        for c in uniq:
            words=set(c["blob"].split())
            kw=len(qset & words)/max(1,len(qset))
            fam=1.0 if c.get("family","") in famset else 0.0
            cwe_match=1.0 if any(c["identifier"] in CWE_TO_NIST.get(x,[]) for x in cwes) else 0.0
            cpe_match=1.0 if c.get("family","") in fam_cpe else 0.0
            score=composite_score(kw,fam,cwe_match,cpe_match,confidence,asset_w,sev_w)
            scored.append((score,c,{"kw_overlap":kw,"family_bonus":fam,"cwe_match":cwe_match,"cpe_match":cpe_match}))
        scored.sort(key=lambda x:x[0],reverse=True)
        top=scored[:6]
        if not top:
            base=nist_df[nist_df["identifier"]=="RA-5"]
            if not base.empty:
                row=base.iloc[0].to_dict()
                top=[(0.35,row,{"kw_overlap":0,"family_bonus":0,"cwe_match":0,"cpe_match":0})]

        controls=[]
        nist_ids=[]
        any_non=False
        top_score=top[0][0] if top else 0
        for score,ctrl,parts in top:
            cid=ctrl["identifier"]; nist_ids.append(cid)
            e=ent_ctrl.get(cid,{})
            impl=e.get("implementation_status","Unknown")
            eff=float(e.get("effectiveness_rating",0.5))
            owner=e.get("owner",""); evid=e.get("evidence","")
            sig=risk_signal(confidence,eff,asset_w)
            posture=classify_posture_by_risk_signal(sig)
            if posture=="Non-Compliant": any_non=True
            controls.append({
                "control_id":cid,"control_name":ctrl.get("name",""),
                "family":ctrl.get("family",""),"score":score,"explain_parts":parts,
                "enterprise_impl_status":impl,"enterprise_effectiveness":eff,
                "enterprise_owner":owner,"enterprise_evidence":evid,
                "risk_signal":sig,"posture":posture,
                "control_text":(ctrl.get("control_text","") or "")[:600]
            })
        refs=enrich_iso_pci(nist_ids)
        action=SEVERITY_ACTIONS.get(severity,"Prioritize remediation.")
        explanation=(f"Controls ranked by composite_score=(0.4*kw+0.2*fam+0.2*cwe+0.2*cpe)*(confidence)*(asset)*(severity). "
                     f"Severity={severity}, confidence={confidence:.2f}, asset_weight={asset_w:.2f}.")
        need_human=requires_human(severity,confidence,asset_w,top_score,any_non)
        packets.append({
            "intel_id":intel_id,"predicted_severity":severity,"confidence":round(confidence,3),
            "recommended_actions":action,"controls":controls,
            "referenced_frameworks":{"NIST":nist_ids,**refs},
            "explanation":explanation,"requires_human":need_human,
            "timestamp":datetime.utcnow().isoformat()
        })

    # Save JSONL + CSV + HTML
    out_jsonl=DIR_OUTPUT/"governance_action_packets_v3.jsonl"
    with open(out_jsonl,"w",encoding="utf-8") as f:
        for p in packets: f.write(json.dumps(p)+"\n")

    flat=[]
    for p in packets:
        nist_ctrls=";".join([c["control_id"] for c in p["controls"][:4]])
        flat.append({
            "intel_id":p["intel_id"],"predicted_severity":p["predicted_severity"],
            "confidence":p["confidence"],"requires_human":p["requires_human"],
            "recommended_actions":p["recommended_actions"],"nist_controls":nist_ctrls,
            "timestamp":p["timestamp"]
        })
    out_csv=DIR_OUTPUT/"governance_action_packets_v3.csv"
    pd.DataFrame(flat).to_csv(out_csv,index=False)

    out_html=DIR_REPORTS/"governance_enterprise_summary_v3.html"
    html=["<html><head><meta charset='utf-8'><title>Governance Summary</title>",
          "<style>body{font-family:Inter,Arial;margin:30px;background:#f9fafb}h3{margin-bottom:6px}</style></head><body>",
          f"<h1>Governance Action Packets — v3</h1><p>Generated {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}</p>"]
    for p in packets:
        overall=max([c["posture"] for c in p["controls"]], key=[c["posture"] for c in p["controls"]].count)
        html.append(f"<div style='background:#fff;padding:12px;margin:8px 0;border-left:5px solid #0d9488'><h3>{p['intel_id']} ({overall})</h3>")
        html.append(f"<p>Severity: {p['predicted_severity']} | Confidence: {p['confidence']:.2f} | Requires human: {p['requires_human']}</p>")
        html.append(f"<p>{p['recommended_actions']}</p></div>")
    html.append("</body></html>")
    out_html.write_text("".join(html),encoding="utf-8")

    print("✓ Component 2 v3 completed successfully.")
    print(f"  JSONL: {out_jsonl}")
    print(f"  CSV  : {out_csv}")
    print(f"  HTML : {out_html}")

if __name__=="__main__":
    main()

✓ Component 2 v3 completed successfully.
  JSONL: /Users/jeevandhamala/Research/AI_GRC_Project/outputs/intel_objects/governance_action_packets_v3.jsonl
  CSV  : /Users/jeevandhamala/Research/AI_GRC_Project/outputs/intel_objects/governance_action_packets_v3.csv
  HTML : /Users/jeevandhamala/Research/AI_GRC_Project/outputs/reports/governance_enterprise_summary_v3.html


In [3]:
# ============================================================
# File: 02_component2_policy_alignment_v3.py
#
# Description:
#   Component 2 — Policy Alignment and Explainable Decision Mapping (v3)
#   Interprets AI_Risk_Intel as governance decision candidates, maps to NIST,
#   consults enterprise internal ruleset, applies asset criticality, computes
#   composite score and proactive risk_signal, then emits Governance_Action_Packets.
# ============================================================

from __future__ import annotations

import os, re, csv, json, logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Dynamic project root detection (for portability)
# ------------------------------------------------------------
def find_project_root() -> Path:
    """Locate the project root dynamically for macOS M2, Linux, or Colab."""
    candidates = [
        Path.cwd(),
        Path.home() / "Research" / "AI_GRC_Project",
        Path("/content/AI_GRC_Project"),  # Colab default path
    ]
    for c in candidates:
        # Find first folder containing either outputs/, data/, or config/
        if (c / "outputs").exists() or (c / "data").exists() or (c / "config").exists():
            return c.resolve()
    # Fallback: current working directory
    return Path.cwd().resolve()

# Automatically resolve project paths
PROJECT_ROOT = find_project_root()
PATH_NIST    = PROJECT_ROOT / "NIST_SP-800-53_rev5_catalog_load.csv"
DIR_OUTPUT   = PROJECT_ROOT / "outputs" / "intel_objects"
DIR_REPORTS  = PROJECT_ROOT / "outputs" / "reports"
DIR_LOGS     = PROJECT_ROOT / "outputs" / "logs"
DIR_CONFIG   = PROJECT_ROOT / "config"

# Ensure all necessary directories exist
for d in (DIR_OUTPUT, DIR_REPORTS, DIR_LOGS, DIR_CONFIG):
    d.mkdir(parents=True, exist_ok=True)

# Configure persistent logging
logging.basicConfig(
    filename=str(DIR_LOGS / "component2_policy_alignment_v3.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("component2_v3")

# ------------------------------------------------------------
# 2. Constants / Policy Weights
# ------------------------------------------------------------
# Define remediation recommendations based on severity
SEVERITY_ACTIONS = {
    "Low":    "Patch in next maintenance window; verify via routine scan (≤ 30 days).",
    "Medium": "Prioritize remediation in the next sprint; retest on completion (≤ 14 days).",
    "High":   "Emergency patch/mitigation; isolate if needed; retest (≤ 72 hours).",
}
# Assign relative severity weighting multipliers
SEVERITY_WEIGHTS = {"Low": 0.8, "Medium": 1.0, "High": 1.2}

# Curated lookup dictionaries connecting CWE, CPE, and framework mappings
CWE_TO_NIST = {
    "CWE-79":  ["SI-10", "SC-7", "RA-5"],
    "CWE-89":  ["SI-10", "RA-5", "SA-11"],
    "CWE-352": ["AC-6", "IA-2", "SI-10"],
    "CWE-22":  ["SC-7", "SI-10"],
    "CWE-78":  ["SI-10", "SI-3", "RA-5"],
    "CWE-119": ["SI-2", "SA-11", "RA-5"],
    "CWE-287": ["IA-2", "AC-6"],
    "CWE-285": ["AC-3", "AC-6"],
    "CWE-798": ["IA-5", "SC-12", "CM-6"],
    "CWE-200": ["SC-7", "SC-28", "CM-2"],
}
# Fallback mappings for plain-text matches in vulnerability descriptions
CWE_HINTS = {
    "xss": ["SI-10", "SC-7", "RA-5"],
    "sql injection": ["SI-10", "RA-5", "SA-11"],
    "command injection": ["SI-10", "SI-3", "RA-5"],
    "buffer overflow": ["SI-2", "SA-11"],
    "csrf": ["AC-6", "IA-2", "SI-10"],
    "authentication": ["IA-2", "AC-6"],
    "authorization": ["AC-3", "AC-6"],
    "path traversal": ["SC-7", "SI-10"],
}
# Vendor / platform level hints to link to specific control families
CPE_HINTS = {
    "microsoft": ["AC", "SC", "SI"],
    "wordpress": ["SI", "AC"],
    "apache": ["SC", "SI"],
    "linux": ["SI", "CM"],
    "cisco": ["SC", "AC"],
}
# Control family cross-mapping to ISO 27001 and PCI-DSS
NIST_TO_ISO = {
    "AC": ["ISO 27001 A.9 (Access Control)"],
    "IA": ["ISO 27001 A.9.4 (Access Control)"],
    "SC": ["ISO 27001 A.13 (Network Security)"],
    "SI": ["ISO 27001 A.12 (Operations Security)"],
    "RA": ["ISO 27001 A.12.6.1 (Vulnerability Management)"],
    "CM": ["ISO 27001 A.12 (Change Management)"],
    "SA": ["ISO 27001 A.14 (Secure Development)"],
}
NIST_TO_PCI = {
    "AC": ["PCI DSS Req. 7 (Access Control)"],
    "IA": ["PCI DSS Req. 8 (Identify/Authenticate)"],
    "SC": ["PCI DSS Req. 1 (Network Security)"],
    "SI": ["PCI DSS Req. 5/6 (Malware/Hardening)"],
    "RA": ["PCI DSS Req. 11 (Testing/Scanning)"],
    "CM": ["PCI DSS Req. 6 (Change Management)"],
    "SA": ["PCI DSS Req. 6 (Secure SDLC)"],
}

# ------------------------------------------------------------
# 3. Loader functions (NIST, Intel, Enterprise config)
# ------------------------------------------------------------
def load_latest_ai_risk_intel() -> List[Dict[str, Any]]:
    """Find and load the newest AI_Risk_Intel JSONL file."""
    cands = sorted(DIR_OUTPUT.glob("ai_risk_intel_*.jsonl"))
    if not cands:
        raise FileNotFoundError(f"No ai_risk_intel_*.jsonl found in {DIR_OUTPUT}")
    latest = max(cands, key=lambda p: p.stat().st_mtime)
    rows = [json.loads(l) for l in latest.read_text(encoding="utf-8").splitlines() if l.strip()]
    logger.info("Loaded %d AI_Risk_Intel records from %s", len(rows), latest.name)
    return rows

def load_nist_catalog(path: Path) -> pd.DataFrame:
    """Load and normalize NIST control catalog CSV."""
    if not path.exists():
        raise FileNotFoundError(f"NIST CSV not found: {path}")
    df = pd.read_csv(path)
    req = ["identifier","name","control_text","discussion","related"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"NIST CSV missing columns: {miss}")
    df = df[req].copy().fillna("")
    def norm(s): return re.sub(r"[^a-z0-9]+"," ",s.lower()).strip()
    # Create single searchable text field
    df["blob"] = (df["identifier"].map(norm)+" "+df["name"].map(norm)+" "+
                  df["control_text"].map(norm)+" "+df["discussion"].map(norm)+" "+
                  df["related"].map(norm))
    # Extract family prefix (e.g., AC, IA, SI)
    df["family"] = df["identifier"].str.extract(r"^([A-Z]{2})", expand=False).fillna("")
    logger.info("Loaded %d NIST controls", len(df))
    return df

def load_enterprise_controls(path: Path) -> Dict[str, Dict[str, Any]]:
    """Load internal enterprise control effectiveness/implementation dataset."""
    if not path.exists():
        logger.warning("enterprise_controls.json not found; continuing without enterprise context.")
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    ctrls = data.get("controls", [])
    return {c["control_id"]: c for c in ctrls if "control_id" in c}

def load_asset_weight(path: Path) -> float:
    """Load enterprise-wide average asset criticality weighting."""
    if not path.exists():
        logger.warning("asset_criticality.json not found; default weight = 0.7")
        return 0.7
    data = json.loads(path.read_text(encoding="utf-8"))
    assets = data.get("assets", [])
    if not assets:
        return 0.7
    return float(np.mean([float(a.get("criticality", 0.7)) for a in assets]))

# ------------------------------------------------------------
# 4. Helper functions for parsing and scoring
# ------------------------------------------------------------
def tok(s: str) -> List[str]:
    """Simple tokenizer (alphanumeric lowercase tokens)."""
    return re.sub(r"[^a-z0-9]+"," ",str(s).lower()).split()

def extract_cwes(rec: Dict[str, Any]) -> List[str]:
    """Extract all CWE identifiers from a record."""
    vals=[]
    for k in ("cwe","cwe_id","cwe_ids","cwes","cwe_list"):
        v=rec.get(k)
        if not v: continue
        if isinstance(v,str): vals+=re.findall(r"CWE-\d+",v.upper())
        elif isinstance(v,list):
            for it in v: vals+=re.findall(r"CWE-\d+",str(it).upper())
    seen=set(); out=[]
    for cid in vals:
        if cid not in seen: seen.add(cid); out.append(cid)
    return out

def extract_cpe_tokens(rec: Dict[str, Any]) -> List[str]:
    """Extract vendor/platform tokens from CPE metadata."""
    allv=[]
    for k in ("cpe","cpe23","cpe_list","cpe_matches","cpe_uris"):
        v=rec.get(k)
        if not v: continue
        allv += [v] if isinstance(v,str) else [str(x) for x in v]
    toks=set()
    for c in allv: toks.update(tok(c))
    return list(toks)

def families_from_cwe_features(cwes: List[str], features: List[str]) -> List[str]:
    """Infer NIST families from CWE categories and key features."""
    fams=[]
    for cid in cwes: fams+=CWE_TO_NIST.get(cid,[])
    if not fams and features:
        joined=" ".join(features).lower()
        for hint,ctrls in CWE_HINTS.items():
            if hint in joined: fams+=ctrls
    out,seen=[],set()
    for ctrl in fams:
        m=re.match(r"^([A-Z]{2})",ctrl)
        if m and m.group(1) not in seen:
            seen.add(m.group(1)); out.append(m.group(1))
    return out[:4]

def families_from_cpe_tokens(tokens: List[str]) -> List[str]:
    """Infer control families from vendor/platform hints."""
    fams,seen=[],set()
    for t in tokens:
        if t in CPE_HINTS:
            for f in CPE_HINTS[t]:
                if f not in seen:
                    seen.add(f); fams.append(f)
    return fams[:4]

# NIST Index class performs lexical similarity search over control text
class NISTIndex:
    def __init__(self, df: pd.DataFrame): self.df=df
    def search_by_tokens(self, tokens: List[str], top_k: int=30) -> List[Dict[str,Any]]:
        """Lexical search using overlap ratio."""
        q={t for t in tokens if t}
        if not q: return []
        scored=[]
        for i,row in self.df.iterrows():
            words=set(row["blob"].split())
            ov=len(q & words)/max(1,len(q))
            if ov>0: scored.append((ov,i))
        scored.sort(reverse=True)
        idx=[i for _,i in scored[:top_k]]
        return self.df.iloc[idx].to_dict(orient="records")
    def by_family(self, fams: List[str], top_k:int=40)->List[Dict[str,Any]]:
        """Filter NIST controls by control family prefix (AC, IA, SI, etc.)."""
        if not fams: return []
        sub=self.df[self.df["family"].isin([f.upper() for f in fams])]
        return sub.head(top_k).to_dict(orient="records")

# Compute composite score for each candidate control
def composite_score(kw_overlap,fam_bonus,cwe_match,cpe_match,confidence,asset_weight,sev_weight):
    base=(0.4*kw_overlap)+(0.2*fam_bonus)+(0.2*cwe_match)+(0.2*cpe_match)
    conf_scale=0.5+0.5*max(0,min(1,confidence))
    asset_scale=0.6+0.4*max(0,min(1,asset_weight))
    return round(base*conf_scale*asset_scale*sev_weight,3)

def risk_signal(confidence,effectiveness,asset_weight):
    """Higher risk_signal = high AI confidence + low control effectiveness + high asset criticality."""
    return round(confidence*(1.0-effectiveness)*asset_weight,3)

def classify_posture_by_risk_signal(sig):
    """Translate risk signal into posture classification."""
    if sig<0.25: return "Compliant"
    if sig<0.60: return "At Risk"
    return "Non-Compliant"

def requires_human(severity,confidence,asset_weight,top_score,any_noncompliant):
    """Decide if human review is mandatory."""
    if severity.lower()=="high": return True
    if confidence<0.85: return True
    if any_noncompliant: return True
    if asset_weight>=0.85 and top_score>=0.60: return True
    return False

def enrich_iso_pci(nist_ids: List[str]) -> Dict[str, List[str]]:
    """Map NIST controls to ISO 27001 and PCI-DSS families."""
    fams=set()
    for cid in nist_ids:
        m=re.match(r"^([A-Z]{2})",cid)
        if m: fams.add(m.group(1))
    iso=[r for f in fams for r in NIST_TO_ISO.get(f,[])]
    pci=[r for f in fams for r in NIST_TO_PCI.get(f,[])]
    return {"ISO27001":list(dict.fromkeys(iso)),"PCI-DSS":list(dict.fromkeys(pci))}

# ------------------------------------------------------------
# 5. Main pipeline: load, interpret, map, export
# ------------------------------------------------------------
def main()->None:
    # Step 1: Load prerequisite data sources
    ai_intel=load_latest_ai_risk_intel()
    nist_df=load_nist_catalog(PATH_NIST)
    idx=NISTIndex(nist_df)
    ent_ctrl=load_enterprise_controls(DIR_CONFIG/"enterprise_controls.json")
    asset_w=load_asset_weight(DIR_CONFIG/"asset_criticality.json")

    packets=[]  # list of Governance_Action_Packets to output

    # Step 2: Process each AI_Risk_Intel record
    for rec in ai_intel:
        intel_id=rec.get("intel_id","")
        severity=(rec.get("predicted_severity") or "Medium").title()
        sev_w=SEVERITY_WEIGHTS.get(severity,1.0)
        dist=rec.get("prediction_distribution") or {}
        confidence=float(max(dist.values())) if dist else 1.0
        features=rec.get("key_influential_features") or rec.get("key_features") or []
        desc=rec.get("description") or rec.get("summary") or ""

        # Extract context (CWE, CPE, families)
        cwes=extract_cwes(rec)
        cpe_toks=extract_cpe_tokens(rec)
        fam_cwe=families_from_cwe_features(cwes,features)
        fam_cpe=families_from_cpe_tokens(cpe_toks)
        fams=list(dict.fromkeys([*fam_cwe,*fam_cpe]))

        # Build search token set for lexical retrieval
        q_tokens=set()
        for f in features: q_tokens.update(tok(f))
        for cw in cwes: q_tokens.update(tok(cw))
        for t in cpe_toks: q_tokens.add(t)
        q_tokens.update(tok(desc)[:16])
        q_tokens=list(q_tokens)

        # Step 3: Candidate retrieval — lexical + family filtering
        cand=idx.search_by_tokens(q_tokens,top_k=40)
        if fams: cand+=idx.by_family(fams,top_k=40)

        # Deduplicate controls by ID
        seen=set(); uniq=[]
        for c in cand:
            cid=c["identifier"]
            if cid not in seen:
                seen.add(cid); uniq.append(c)

        # Step 4: Scoring each candidate control
        scored=[]
        qset,famset=set(q_tokens),set(fams)
        for c in uniq:
            words=set(c["blob"].split())
            kw=len(qset & words)/max(1,len(qset))
            fam=1.0 if c.get("family","") in famset else 0.0
            cwe_match=1.0 if any(c["identifier"] in CWE_TO_NIST.get(x,[]) for x in cwes) else 0.0
            cpe_match=1.0 if c.get("family","") in fam_cpe else 0.0
            score=composite_score(kw,fam,cwe_match,cpe_match,confidence,asset_w,sev_w)
            scored.append((score,c,{"kw_overlap":kw,"family_bonus":fam,"cwe_match":cwe_match,"cpe_match":cpe_match}))
        scored.sort(key=lambda x:x[0],reverse=True)
        top=scored[:6]
        if not top:
            # Fallback to generic RA-5 vulnerability scanning if nothing matches
            base=nist_df[nist_df["identifier"]=="RA-5"]
            if not base.empty:
                row=base.iloc[0].to_dict()
                top=[(0.35,row,{"kw_overlap":0,"family_bonus":0,"cwe_match":0,"cpe_match":0})]

        # Step 5: Compute enterprise posture + risk signal for each control
        controls=[]; nist_ids=[]; any_non=False
        top_score=top[0][0] if top else 0
        for score,ctrl,parts in top:
            cid=ctrl["identifier"]; nist_ids.append(cid)
            e=ent_ctrl.get(cid,{})
            impl=e.get("implementation_status","Unknown")
            eff=float(e.get("effectiveness_rating",0.5))
            sig=risk_signal(confidence,eff,asset_w)
            posture=classify_posture_by_risk_signal(sig)
            if posture=="Non-Compliant": any_non=True
            controls.append({
                "control_id":cid,"control_name":ctrl.get("name",""),
                "family":ctrl.get("family",""),"score":score,"explain_parts":parts,
                "enterprise_impl_status":impl,"enterprise_effectiveness":eff,
                "risk_signal":sig,"posture":posture,
                "control_text":(ctrl.get("control_text","") or "")[:600]
            })

        # Step 6: Reference frameworks and decision summary
        refs=enrich_iso_pci(nist_ids)
        action=SEVERITY_ACTIONS.get(severity,"Prioritize remediation.")
        explanation=(f"Controls ranked by composite_score=(0.4*kw+0.2*fam+0.2*cwe+0.2*cpe)*(confidence)*(asset)*(severity). "
                     f"Severity={severity}, confidence={confidence:.2f}, asset_weight={asset_w:.2f}.")
        need_human=requires_human(severity,confidence,asset_w,top_score,any_non)

        # Create Governance_Action_Packet record
        packets.append({
            "intel_id":intel_id,"predicted_severity":severity,"confidence":round(confidence,3),
            "recommended_actions":action,"controls":controls,
            "referenced_frameworks":{"NIST":nist_ids,**refs},
            "explanation":explanation,"requires_human":need_human,
            "timestamp":datetime.utcnow().isoformat()
        })

    # ------------------------------------------------------------
    # Step 7: Save governance artifacts (JSONL, CSV, HTML)
    # ------------------------------------------------------------
    out_jsonl=DIR_OUTPUT/"governance_action_packets_v3.jsonl"
    with open(out_jsonl,"w",encoding="utf-8") as f:
        for p in packets: f.write(json.dumps(p)+"\n")

    flat=[]
    for p in packets:
        nist_ctrls=";".join([c["control_id"] for c in p["controls"][:4]])
        flat.append({
            "intel_id":p["intel_id"],"predicted_severity":p["predicted_severity"],
            "confidence":p["confidence"],"requires_human":p["requires_human"],
            "recommended_actions":p["recommended_actions"],"nist_controls":nist_ctrls,
            "timestamp":p["timestamp"]
        })
    out_csv=DIR_OUTPUT/"governance_action_packets_v3.csv"
    pd.DataFrame(flat).to_csv(out_csv,index=False)

    # Minimal HTML visualization for executive review
    out_html=DIR_REPORTS/"governance_enterprise_summary_v3.html"
    html=[
        "<html><head><meta charset='utf-8'><title>Governance Summary</title>",
        "<style>body{font-family:Inter,Arial;margin:30px;background:#f9fafb}h3{margin-bottom:6px}</style></head><body>",
        f"<h1>Governance Action Packets — v3</h1><p>Generated {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}</p>"
    ]
    for p in packets:
        # Determine majority control posture
        overall=max([c["posture"] for c in p["controls"]], key=[c["posture"] for c in p["controls"]].count)
        html.append(f"<div style='background:#fff;padding:12px;margin:8px 0;border-left:5px solid #0d9488'><h3>{p['intel_id']} ({overall})</h3>")
        html.append(f"<p>Severity: {p['predicted_severity']} | Confidence: {p['confidence']:.2f} | Requires human: {p['requires_human']}</p>")
        html.append(f"<p>{p['recommended_actions']}</p></div>")
    html.append("</body></html>")
    out_html.write_text("".join(html),encoding="utf-8")

    print("✓ Component 2 v3 completed successfully.")
    print(f"  JSONL: {out_jsonl}")
    print(f"  CSV  : {out_csv}")
    print(f"  HTML : {out_html}")

# ------------------------------------------------------------
# Entrypoint
# ------------------------------------------------------------
if __name__=="__main__":
    main()

✓ Component 2 v3 completed successfully.
  JSONL: /Users/jeevandhamala/Research/AI_GRC_Project/outputs/intel_objects/governance_action_packets_v3.jsonl
  CSV  : /Users/jeevandhamala/Research/AI_GRC_Project/outputs/intel_objects/governance_action_packets_v3.csv
  HTML : /Users/jeevandhamala/Research/AI_GRC_Project/outputs/reports/governance_enterprise_summary_v3.html
